# ¿Qué hace que un juego en Steam sea popular?

## 1. Introducción

### 1.1 Contexto

### 1.2 Objetivo

### 1.3 Variable Dependiente

### 1.4 Variable Independiente

### 1.5 Setup

In [ ]:
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import Markdown, display

import statsmodels.api as sm

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

RANDOM_STATE = 2026
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Base de Datos

### 2.1 Fuente

Martin Bustos Roman. (2022). Steam Games Dataset [Dataset]. Kaggle. https://doi.org/10.34740/KAGGLE/DS/2109585

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fronkongames/steam-games-dataset")
print("Path to dataset files:", path)

path_to_csv = os.path.join(path, "games.csv")
print("Path to csv file:", path_to_csv)

### 2.2 Tamaño

In [ ]:
# Loading the dataset and having a glimpse
raw_dataset = pd.read_csv(path_to_csv,
                   sep=','
                   )
print(f"Dimensions: {raw_dataset.shape}")

In [ ]:
# Taking a look at the head of the dataset
raw_dataset.head()

Antes de empezar todo el analisis que viene a continuación, fue necesario corregir el header, pues dos variables venian unidas, porque le faltaba una coma.

Esto puede ser corroborado arriba.

- DiscountDLC count -> Discount, DLC count

De esta manera tendríamos las 40 variables iniciales.

In [ ]:
# From a quick look and reading the discussion section in kaggle
# we notice there seems to be a comma missing between the header
# Discount, DLC count
# Applying the fix...
header = raw_dataset.columns.to_list()
print(f"Old Header: {header}")

fixed_header = header[:7] + ["Discount", "DLC count"] + header[8:]
fixed_dataset = pd.read_csv(path_to_csv,
                            na_values="?",
                            sep=',',
                            names=fixed_header,
                            header=0
                            )
print(f"Fixed Dimensions: {fixed_dataset.shape}")
fixed_dataset.head()

### 2.3 Análisis

In [ ]:
# Initial statistics of the data
fixed_dataset.describe().T

falta texto, notar que es analisis de la base de datos, en general

### 2.4 Diccionario de Variables

#### 2.4.1 Tipos de Dato

In [ ]:
# Checking the data
fixed_dataset.info()

#### 2.4.2 Significado

significado de las variables...


## 3. Limpieza y Preparación

### 3.1 Suposiciones de Data

#### 3.1.1 Missing Values

Antes de avanzar debemos de revisar cuanta de la data es vacia (NA) y cuanta de ella es realmente NA disfrazada, por ejemplo, "[]" y ceros en algunas columnas.

Hay que tener cuidado de no tomar cualquier 0 como NA, por ejemplo, precio 0 quiere decir gratis y no que este vacio.

In [ ]:
def missing_summary_table(df):
    """
    Returns a summary table of missing values in the dataset.
    """
    # Each measure is a Series: index = column name, value = % of rows
    numeric_cols = df.select_dtypes(include="number")
    zero_pct = (numeric_cols == 0).mean() * 100

    null_pct = df.isna().mean() * 100

    empty_list_pct = df.isin(["[]"]).mean() * 100

    summary = pd.DataFrame({
        "% nulos": null_pct,
        "% lista vacía '[]'": empty_list_pct,
        "% ceros": zero_pct,
    })

    summary["% vacío total"] = summary["% nulos"] + summary["% lista vacía '[]'"]
    return summary.sort_values(["% vacío total", "% ceros"], ascending=False).round(2)

missing_summary_table(fixed_dataset)

In [ ]:
# Check: is "Metacritic score == 0" the same as "no Metacritic URL"?
score_is_zero = fixed_dataset["Metacritic score"] == 0
url_is_missing = fixed_dataset["Metacritic url"].isna()

pd.crosstab(
    score_is_zero,
    url_is_missing,
    rownames=["Metacritic score = 0"],
    colnames=["Metacritic url vacío"],
)

- Al analizar los datos vacios podemos decir que definitivamente la columna Movies se puede botar, pues no contiene nada de data. 
- Otro candidato para ser botado es Score rank.
- Metacritic score = 0 significa "sin reseña de crítica", no una calificación de 0.

Otras suposiciones:
- Price 0 = gratis
- Required age 0 = todo el publico.

#### 3.1.2 Juegos sin Actividad

In [ ]:
# Check: do games with "0 - 0" owners have any reviews?
no_owners = fixed_dataset["Estimated owners"] == "0 - 0"
games_with_reviews = fixed_dataset["Positive"] + fixed_dataset["Negative"] > 0

pd.crosstab(
    no_owners,
    games_with_reviews,
    rownames=["Owners = 0 - 0"],
    colnames=["Juegos con reviews"],
)

De aquí se podría decir que 24579 son juegos sin actividad registrada (playtests, próximos lanzamientos o sin datos), por lo que no aporta a la pregunta de que determina la popularidad de un juego. Por lo tanto, estos juegos pueden ser botados.

### 3.2 Modificaciones

#### 3.2.1 Juegos sin Actividad

In [ ]:
# Keep only games with registered activity (owners other than "0 - 0")
has_activity = fixed_dataset["Estimated owners"] != "0 - 0"
released_games = fixed_dataset[has_activity].copy()

print(f"Antes:      {fixed_dataset.shape}")
print(f"Después:    {released_games.shape}")
print(f"Eliminados: {len(fixed_dataset) - len(released_games):,} juegos")

In [ ]:
# Compare % empty per column before and after removing inactive games
before = missing_summary_table(fixed_dataset)["% vacío total"]
after = missing_summary_table(released_games)["% vacío total"]

empty_comparison = pd.DataFrame({
    "% vacío antes": before,
    "% vacío después": after,
})
empty_comparison["diferencia"] = empty_comparison["% vacío después"] - empty_comparison["% vacío antes"]

# Only columns that had something empty, largest first
empty_comparison[empty_comparison["% vacío antes"] > 0].sort_values("% vacío antes", ascending=False)

Solo quitando aquellos juegos sin actividad se logro limpiar bastantes los datos vacios. Ahora un ultimo paso...

#### 3.2.2 Eliminación de Columnas

- Sin información: Movies (100% vacio) y Score rank (Casi el 100% vacio)
- Identificadores: AppID
- Leakage: Basicamente miden lo mismo que el target, entonces en la matriz de correlación se presentarian con correlaciones altas, porque es lo mismo que el target. Los features que quedan satisfacen la pregunta: 
  Cuales variables estaban antes de que el juego sea publicado?

In [ ]:
# Columns dropped by criterion (see explanation above)
no_information = ["Movies", "Score rank"]
identifiers = ["AppID"]
leakage = [
    "Peak CCU", "Estimated owners", "Recommendations",
    "Average playtime forever", "Average playtime two weeks",
    "Median playtime forever", "Median playtime two weeks",
    "Tags",
]

columns_to_drop = no_information + identifiers + leakage
cleaned_dataset = released_games.drop(columns=columns_to_drop)

print(f"Columnas: {released_games.shape[1]} → {cleaned_dataset.shape[1]}")
print(f"Eliminadas: {len(columns_to_drop)}")

#### 3.2.3 Valores Vacíos e Imputación 

llenar aqui de info de los tipos de imputaciones

In [ ]:
# What is still empty after the previous steps?
remaining = missing_summary_table(cleaned_dataset)
remaining[(remaining["% vacío total"] > 0) | (remaining["% ceros"] > 90)]

In [ ]:
# Missing indicator: 1 if critics reviewed the game, 0 otherwise
cleaned_dataset["has_metacritic"] = (cleaned_dataset["Metacritic score"] > 0).astype(int)

# The score and URL are now redundant
cleaned_dataset = cleaned_dataset.drop(columns=["Metacritic score", "Metacritic url"])

cleaned_dataset["has_metacritic"].value_counts()

#### 3.2.4 Variable Objetivo

In [ ]:
# Target: total number of reviews (proxy for popularity)
cleaned_dataset["n_reviews"] = cleaned_dataset["Positive"] + cleaned_dataset["Negative"]
cleaned_dataset["log_reviews"] = np.log1p(cleaned_dataset["n_reviews"])

cleaned_dataset[["n_reviews", "log_reviews"]].describe().T

Por que se necesita el uso de log?

In [ ]:
# Why is the log needed?
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(cleaned_dataset["n_reviews"], bins=50)
axes[0].set_title(f"Reviews (asimetría = {cleaned_dataset['n_reviews'].skew():.1f})")

axes[1].hist(cleaned_dataset["log_reviews"], bins=50)
axes[1].set_title(f"log(1 + reviews) (asimetría = {cleaned_dataset['log_reviews'].skew():.2f})")

plt.tight_layout()
plt.show()

In [ ]:
# Dropping the original reviews column, we will use the log version as target
cleaned_dataset = cleaned_dataset.drop(columns=["Positive", "Negative", "n_reviews"])
cleaned_dataset.describe().T

#### 3.2.5 Construcción de Variables

In [ ]:
# Presence indicators: 1 if the field has content, 0 if it is empty
presence_columns = {
    "has_website": "Website",
    "has_support_url": "Support url",
    "has_support_email": "Support email",
    "has_press_quotes": "Reviews",
    "has_notes": "Notes",
    "has_header_image": "Header image",
}

for new_column, source_column in presence_columns.items():
    cleaned_dataset[new_column] = cleaned_dataset[source_column].notna().astype(int)

cleaned_dataset = cleaned_dataset.drop(columns=list(presence_columns.values()))

In [ ]:
# Mature content: Required age > 0
cleaned_dataset["mature"] = (cleaned_dataset["Required age"] > 0).astype(int)
cleaned_dataset = cleaned_dataset.drop(columns=["Required age"])

In [ ]:
# Platforms: bool -> int
cleaned_dataset[["Windows", "Mac", "Linux"]] = cleaned_dataset[["Windows", "Mac", "Linux"]].astype(int)

In [ ]:
binary_columns = list(presence_columns) + ["mature", "Windows", "Mac", "Linux"]
(cleaned_dataset[binary_columns].mean() * 100).round(2).sort_values()

##### Conteos y longitudes

In [ ]:
def count_list_items(value):
    """Number of items in a comma-separated list stored as text ('[]' or NaN -> 0)."""
    if pd.isna(value) or value == "[]":
        return 0
    return value.count(",") + 1

count_columns = {
    "n_languages": "Supported languages",
    "n_audio_languages": "Full audio languages",
    "n_screenshots": "Screenshots",
}

for new_column, source_column in count_columns.items():
    cleaned_dataset[new_column] = cleaned_dataset[source_column].map(count_list_items)

# Does the game have English voice acting?
cleaned_dataset["english_audio"] = (
    cleaned_dataset["Full audio languages"].str.contains("'English'", regex=False).astype(int)
)

In [ ]:
cleaned_dataset["name_length"] = cleaned_dataset["Name"].fillna("").str.len()
cleaned_dataset["description_length"] = cleaned_dataset["About the game"].fillna("").str.len()
cleaned_dataset = cleaned_dataset.drop(columns=list(count_columns.values()) + ["About the game"])

In [ ]:
new_columns = list(count_columns) + ["english_audio", "name_length", "description_length"]
cleaned_dataset[new_columns].describe().T

##### Antigüedad del juego

In [ ]:
# Release date: text -> datetime (every date follows "Mon D, YYYY", e.g. "Aug 1, 2023")
release = pd.to_datetime(cleaned_dataset["Release date"], format="%b %d, %Y")

# Reference date: the latest release in the data (approximately when it was collected)
reference_date = release.max()
cleaned_dataset["years_since_release"] = (reference_date - release).dt.days / 365.25

cleaned_dataset = cleaned_dataset.drop(columns=["Release date"])

print(f"Fecha de referencia: {reference_date.date()}")
cleaned_dataset["years_since_release"].describe()

##### Géneros y categorías

In [ ]:
MIN_SHARE = 5  # % of games; rarer labels give unstable coefficients

def to_snake_case(label):
    """'Free To Play' -> 'free_to_play', 'Shared/Split Screen' -> 'shared_split_screen'."""
    return label.lower().replace(" ", "_").replace("-", "_").replace("/", "_")

def frequent_dummies(column, prefix, exclude=()):
    """One 0/1 column per label present in at least MIN_SHARE % of games."""
    dummies = cleaned_dataset[column].str.get_dummies(sep=",")
    dummies = dummies.drop(columns=list(exclude))
    share = dummies.mean() * 100
    dummies = dummies.loc[:, share >= MIN_SHARE]
    return dummies.rename(columns=lambda label: prefix + to_snake_case(label))

genre_dummies = frequent_dummies("Genres", "genre_")
category_dummies = frequent_dummies(
    "Categories", "cat_",
    exclude=["Steam Trading Cards",   # leakage: Valve enables them after a game sells
             "Steam Achievements"],   # redundant with the Achievements count
)

cleaned_dataset = pd.concat([cleaned_dataset, genre_dummies, category_dummies], axis=1)
cleaned_dataset = cleaned_dataset.drop(columns=["Genres", "Categories"])

print(f"Géneros: {list(genre_dummies.columns)}")
print(f"Categorías: {list(category_dummies.columns)}")

##### Desarrolladores y publicadores

In [ ]:
developers = cleaned_dataset["Developers"].fillna("")
publishers = cleaned_dataset["Publishers"].fillna("")

# Self-published: the developer is also the publisher (typical of small indie studios)
cleaned_dataset["self_published"] = ((developers == publishers) & (developers != "")).astype(int)

# Publisher catalog size: number of games in the dataset with the same publisher
cleaned_dataset["publisher_n_games"] = publishers.map(publishers.value_counts())
cleaned_dataset.loc[publishers == "", "publisher_n_games"] = 0

cleaned_dataset = cleaned_dataset.drop(columns=["Developers", "Publishers"])
cleaned_dataset[["self_published", "publisher_n_games"]].describe().T

#### 3.2.6 Transformaciones Logarítmicas:

In [ ]:
# Free games: create before transforming Price
cleaned_dataset["is_free"] = (cleaned_dataset["Price"] == 0).astype(int)

# Variables with skewness > 1: log1p compresses the long right tail (0 stays 0)
log_columns = [
    "DLC count", "Achievements", "Price", "description_length",
    "n_audio_languages", "n_languages", "publisher_n_games",
    "n_screenshots", "name_length",
]

skewness = pd.DataFrame({"asimetría antes": cleaned_dataset[log_columns].skew()})

for column in log_columns:
    cleaned_dataset["log_" + to_snake_case(column)] = np.log1p(cleaned_dataset[column])

log_names = ["log_" + to_snake_case(column) for column in log_columns]
skewness["asimetría después"] = cleaned_dataset[log_names].skew().values
cleaned_dataset = cleaned_dataset.drop(columns=log_columns)

skewness.round(2)

In [ ]:
# Final check: everything numeric (except Name) and no missing values
print(f"Dimensiones: {cleaned_dataset.shape}")
print(f"Valores vacíos: {cleaned_dataset.isna().sum().sum()}")
print(f"Columnas no numéricas: {list(cleaned_dataset.select_dtypes(exclude='number').columns)}")

## 4. Análisis Descriptivo y Correlación

### 4.1 Estadísticas Relevantes

In [ ]:
# Binary variables: only 0/1 values. The rest are continuous (Name is text, so it's excluded)
numeric = cleaned_dataset.select_dtypes(include="number")
binary_columns = numeric.columns[numeric.isin([0, 1]).all()].tolist()
continuous_columns = [c for c in numeric.columns if c not in binary_columns]

numeric[continuous_columns].describe().T

In [ ]:
rows = []
for column in binary_columns:
    has_it = cleaned_dataset[column] == 1
    rows.append({
        "variable": column,
        "% juegos": has_it.mean() * 100,
        "mediana reviews (sí)": np.expm1(cleaned_dataset.loc[has_it, "log_reviews"].median()),
        "mediana reviews (no)": np.expm1(cleaned_dataset.loc[~has_it, "log_reviews"].median()),
    })

binary_summary = pd.DataFrame(rows).set_index("variable")
binary_summary.sort_values("mediana reviews (sí)", ascending=False).round(1)

In [ ]:
# Does log_reviews measure popularity? Compare with other popularity measures (not used as predictors)
popularity = pd.DataFrame({
    "log_reviews": cleaned_dataset["log_reviews"],
    "log_peak_ccu": np.log1p(released_games["Peak CCU"]),
    "log_recommendations": np.log1p(released_games["Recommendations"]),
    "log_playtime": np.log1p(released_games["Average playtime forever"]),
})
popularity.corr()["log_reviews"].round(2)

### 4.2 Matriz de Correlación

In [ ]:
# Correlation matrix of the final variables (Name is text, so numeric_only skips it)
corr = cleaned_dataset.corr(numeric_only=True)

# Order by |correlation with the target|: the first column becomes a ranking
order = corr["log_reviews"].abs().sort_values(ascending=False).index
corr = corr.loc[order, order]

plt.figure(figsize=(26, 22))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", annot_kws={"size": 6},
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.3, cbar_kws={"shrink": 0.6},
)
plt.title("Matriz de correlación (ordenada por correlación con log_reviews)")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each variable with the target, as a single-column heatmap
target_corr = corr[["log_reviews"]].drop(index="log_reviews")

plt.figure(figsize=(5, 16))
sns.heatmap(target_corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Correlación con log_reviews")
plt.tight_layout()
plt.show()

### 4.3 Interpretración

- target validation (strong correlation with other popularity measures)
- the correlation strength scale applied to the single-column heatmap
- the "warning" variables (inexistente correlation)
- the redundant predictor pairs
- the caution about one-at-a-time comparisons

Mas detallado en espaNol.

## 5. Selección de Variables

método completo fundamentado hasta el modelo final

### 5.1 División train/test

In [ ]:
# 70/30 split as in the class lab. Variable selection uses only train;
# test is kept untouched until section 6.
train_data, test_data = train_test_split(
    cleaned_dataset, test_size=0.30, random_state=RANDOM_STATE
)

TARGET = "log_reviews"
candidate_features = [c for c in cleaned_dataset.columns if c not in ("Name", TARGET)]

print(f"Train: {train_data.shape[0]:,} juegos | Test: {test_data.shape[0]:,} juegos")
print(f"Variables candidatas: {len(candidate_features)}")

### 5.2 Modelo 1: todas las variables

In [ ]:
def linear_regression(data, target, features):
    X = sm.add_constant(data[features])
    y = data[target]
    modelo = sm.OLS(y, X).fit()
    return modelo

model_1 = linear_regression(train_data, TARGET, candidate_features)
print(model_1.summary())

### 5.3 Backward elimination

In [ ]:
def backward_elimination(data, target, features, alpha=0.05):
    """Remove the least significant variable one at a time until every p-value <= alpha."""
    features = list(features)
    steps = []
    while True:
        model = linear_regression(data, target, features)
        p_values = model.pvalues.drop("const")
        worst = p_values.idxmax()
        done = p_values[worst] <= alpha
        steps.append({
            "n variables": len(features),
            "R²": model.rsquared,
            "R² adj": model.rsquared_adj,
            "F": model.fvalue,
            "eliminada": "-" if done else worst,
            "p-value": p_values[worst],
        })
        if done:
            return model, features, pd.DataFrame(steps)
        features.remove(worst)

model_backward, backward_features, backward_steps = backward_elimination(
    train_data, TARGET, candidate_features
)
backward_steps.round(4)

In [ ]:
# Does Windows change anything? Same final model without it
features_without_windows = [f for f in backward_features if f != "Windows"]
model_without_windows = linear_regression(train_data, TARGET, features_without_windows)

print(f"R² adj con Windows: {model_backward.rsquared_adj:.4f}")
print(f"R² adj sin Windows: {model_without_windows.rsquared_adj:.4f}")

windows_check = pd.DataFrame({
    "con Windows": model_backward.params,
    "sin Windows": model_without_windows.params,
})
windows_check.loc[["const", "Windows", "has_metacritic", "years_since_release", "log_dlc_count"]].round(3)

Aunque el backward elimination dejó a Windows en el modelo porque su valor p salió menor a 0.05, decidimos quitarla. El problema es que casi todos los juegos de Steam corren en Windows (99.97%), así que en realidad el modelo solo tiene unos 20 juegos para "aprender" qué pasa cuando un juego no la tiene. 

Eso no es suficiente para sacar una conclusión confiable. Además, su correlación con los reviews era prácticamente cero, y como vimos en clase, el valor p no debe ser el único criterio. Para comprobarlo, corrimos el modelo sin Windows: el R² ajustado casi no cambió y los demás coeficientes quedaron iguales. 

Lo único que cambió fue el intercepto, lo que nos dice que Windows estaba funcionando como una segunda constante en lugar de aportar algo nuevo al modelo.

In [ ]:
# Decision: drop Windows (see justification above)
final_features = features_without_windows
model_final = model_without_windows

### 5.4 Forward selection

In [ ]:
def forward_selection(data, target, features, alpha=0.05):
    """Add, one at a time, the variable that raises adjusted R² the most, while it is significant."""
    remaining = list(features)
    selected = []
    steps = []
    while remaining:
        # Try every remaining variable and keep the one with the best adjusted R²
        scores = {
            candidate: linear_regression(data, target, selected + [candidate]).rsquared_adj
            for candidate in remaining
        }
        best_candidate = max(scores, key=lambda candidate: scores[candidate])
        best_model = linear_regression(data, target, selected + [best_candidate])

        # Stop when even the best candidate is not significant
        if best_model.pvalues[best_candidate] > alpha:
            break

        selected.append(best_candidate)
        remaining.remove(best_candidate)
        steps.append({
            "n variables": len(selected),
            "agregada": best_candidate,
            "R²": best_model.rsquared,
            "R² adj": best_model.rsquared_adj,
            "F": best_model.fvalue,
            "p-value": best_model.pvalues[best_candidate],
        })

    return linear_regression(data, target, selected), selected, pd.DataFrame(steps)

# Same candidates as backward, without Windows
forward_candidates = [f for f in candidate_features if f != "Windows"]
model_forward, forward_features, forward_steps = forward_selection(
    train_data, TARGET, forward_candidates
)
forward_steps.round(4)

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(forward_steps["n variables"], forward_steps["R² adj"], marker="o")
plt.xlabel("Número de variables")
plt.ylabel("R² ajustado")
plt.title("Forward selection: R² ajustado al agregar variables")
plt.grid(True)
plt.show()

In [ ]:
print(f"Backward (sin Windows): {len(final_features)} variables | R² adj = {model_final.rsquared_adj:.4f}")
print(f"Forward:                {len(forward_features)} variables | R² adj = {model_forward.rsquared_adj:.4f}")
print(f"Solo en backward: {sorted(set(final_features) - set(forward_features))}")
print(f"Solo en forward:  {sorted(set(forward_features) - set(final_features))}")

### 5.5 Comparación y modelo final

In [ ]:
def compare_models(models: dict):
    rows = []
    for name, m in models.items():
        rows.append({
            "Modelo": name,
            "R²": m.rsquared,
            "R² adj": m.rsquared_adj,
            "F": m.fvalue,
            "RMSE (train)": np.sqrt(np.mean(m.resid ** 2)),
            "n_params": len(m.params),
        })
    return pd.DataFrame(rows).set_index("Modelo")

# Reduced model: the first 10 variables chosen by forward selection
reduced_features = forward_features[:10]
model_reduced = linear_regression(train_data, TARGET, reduced_features)

compare_models({
    "Modelo 1 (todas)": model_1,
    "Modelo final (backward = forward)": model_final,
    "Modelo reducido (10 variables)": model_reduced,
}).round(4)

aqui va conclusion

## 6. Train/Test Split

### 6.1 Métricas en test

In [ ]:
# Final model (fitted on train) predicting the test set, which it has never seen
X_test = sm.add_constant(test_data[final_features])
y_test = test_data[TARGET]
y_pred_test = model_final.predict(X_test)

y_train = train_data[TARGET]
y_pred_train = model_final.fittedvalues

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)

print(f"R² train:   {model_final.rsquared:.4f} | R² test:   {r2_test:.4f}")
print(f"RMSE train: {rmse_train:.4f} | RMSE test: {rmse_test:.4f}")
print(f"Ratio RMSE test/train: {rmse_test / rmse_train:.4f}")
print(f"RMSE normalizado (RMSE / desviación estándar): {rmse_test / y_test.std():.4f}")
print(f"Factor de error (e^RMSE): {np.exp(rmse_test):.2f}x")

### 6.2 Validación cruzada

In [ ]:
# 10-fold cross-validation on train, same setup as the class lab
X_train = train_data[final_features]
pipeline = Pipeline([("linear", LinearRegression())])
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

rmse_cv = -cross_val_score(pipeline, X_train, y_train, scoring="neg_root_mean_squared_error", cv=kf)
r2_cv = cross_val_score(pipeline, X_train, y_train, scoring="r2", cv=kf)

print(f"RMSE (10-fold CV): {rmse_cv.mean():.4f} ± {rmse_cv.std():.4f}")
print(f"R²   (10-fold CV): {r2_cv.mean():.4f} ± {r2_cv.std():.4f}")

### 6.3 Predicciones y residuos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, y_pred_test, alpha=0.1, s=4)
lims = [0, max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, "r--", linewidth=1)
axes[0].set_xlabel("log_reviews real")
axes[0].set_ylabel("log_reviews predicho")
axes[0].set_title(f"Test: predicción vs real (RMSE = {rmse_test:.3f})")

resid_test = y_test - y_pred_test
axes[1].scatter(y_pred_test, resid_test, alpha=0.1, s=4)
axes[1].axhline(0, color="red", linestyle="--", linewidth=1)
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Residuo")
axes[1].set_title("Residuos en test")

plt.tight_layout()
plt.show()

## 7. Modelo Final (Regresión Lineal)

### 7.1 Ecuación Estimada

In [ ]:
print(model_final.summary())

In [ ]:
TYPE_ORDER = {"continua": 0, "logarítmica": 1, "binaria": 2}

def variable_type(name):
    """Type of a feature, which decides how its coefficient is read."""
    if name.startswith("log_"):
        return "logarítmica"
    if name in binary_columns:
        return "binaria"
    return "continua"

params = model_final.params
features = sorted(params.index.drop("const"), key=lambda name: TYPE_ORDER[variable_type(name)])
symbol_index = {name: i for i, name in enumerate(features, start=1)}

terms = [f"{params['const']:.3f}"]
for name in features:
    coef = params[name]
    sign = "+" if coef >= 0 else "-"
    terms.append(f"{sign} {abs(coef):.3f}\\,x_{{{symbol_index[name]}}}")

# Six terms per line: the short symbols leave room for more
lines = [" ".join(terms[i:i + 6]) for i in range(0, len(terms), 6)]
equation = r"\begin{aligned}\hat{y} = {} & " + r" \\ & ".join(lines) + r"\end{aligned}"
display(Markdown(f"$$ {equation} $$"))

### 7.2 Interpretación de Coeficientes

In [ ]:
def effect_in_reviews(name, coef):
    """% change in reviews implied by a coefficient of the log target."""
    if variable_type(name) == "logarítmica":
        return (2 ** coef - 1) * 100
    return (np.exp(coef) - 1) * 100

conf_int = model_final.conf_int()
rows = []
for name in features:
    coef = params[name]
    rows.append({
        "símbolo": f"x{symbol_index[name]}",
        "variable": name,
        "tipo": variable_type(name),
        "coeficiente": coef,
        "IC 95% inferior": conf_int.loc[name, 0],
        "IC 95% superior": conf_int.loc[name, 1],
        "p-value": model_final.pvalues[name],
        "efecto en reviews (%)": effect_in_reviews(name, coef),
    })

coefficients = pd.DataFrame(rows).set_index("símbolo")
coefficients.round(3)

In [ ]:
ordered = coefficients.sort_values("coeficiente")
errors = [ordered["coeficiente"] - ordered["IC 95% inferior"],
          ordered["IC 95% superior"] - ordered["coeficiente"]]

plt.figure(figsize=(9, 12))
plt.errorbar(ordered["coeficiente"], ordered["variable"], xerr=errors, fmt="o", capsize=3)
plt.axvline(0, color="red", linestyle="--", linewidth=1)
plt.xlabel("Coeficiente (escala log)")
plt.title("Coeficientes del modelo final con intervalo de confianza 95%")
plt.tight_layout()
plt.show()

La gráfica muestra cada coeficiente con su intervalo de confianza del 95%. Sirve para ver en un vistazo qué variables aumentan o disminuyen los reviews o la popularidad y confirmar que ningún intervalo cruza el cero, lo cual coincide con que todos los p-values son menores a 0.05. 

No sirve para comparar qué variable es más importante, porque las binarias y las logarítmicas están en unidades distintas. 

Por ejemplo, un coeficiente de 0.5 en una binaria significa tener o no tener la característica, mientras que en una logarítmica significa duplicar la variable.

## 8. Modelos Extra

### 8.1 Modelos

In [ ]:
extra_models = {
    "Regresión lineal (OLS)": Pipeline([("linear", LinearRegression())]),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 13))),
    ]),
    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", LassoCV(cv=5, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Gradient Boosting": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
}

### 8.2 Comparación

In [ ]:
# sklearn adds its own intercept, so no constant column here
X_test_plain = test_data[final_features]

def evaluate(model):
    """10-fold CV on train, then fit on all of train and score on test."""
    cv_rmse = -cross_val_score(model, X_train, y_train, scoring="neg_root_mean_squared_error", cv=kf)
    model.fit(X_train, y_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    y_pred = model.predict(X_test_plain)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return {
        "RMSE CV": cv_rmse.mean(),
        "RMSE CV (std)": cv_rmse.std(),
        "RMSE train": train_rmse,
        "RMSE test": test_rmse,
        "Ratio test/train": test_rmse / train_rmse,
        "R² test": r2_score(y_test, y_pred),
    }

model_comparison = pd.DataFrame({name: evaluate(model) for name, model in extra_models.items()}).T
model_comparison.sort_values("RMSE test")

De aqui se puede decir que Lasso y Ridge no cambiaron mucho contra OLS, pues al final también son lineales. En el caso de los modelos de arboles, RF redujo bastante el RMSE igual que Gradient Boosting, pero su ratio es test >> train, lo cual quiere decir que overfiting. A pesar de esto, igual generaliza mejor que OLS.

Con esto, podriamos decir que nuestro mejor modelo hasta este punto es Gradient Boosting.

In [ ]:
lasso = extra_models["Lasso"].named_steps["lasso"]
print(f"Alpha elegido: {lasso.alpha_:.5f}")
print(f"Coeficientes en 0: {(lasso.coef_ == 0).sum()} de {len(lasso.coef_)}")

Si no hay coeficientes en 0, esto quiere decir que coincide con nuestro backward y forward selection y que fue exitoso, y asi lo confirmamos tambien de otra manera.

### 8.3 Kumo Tabular

Queriamos experiementar con un nuevo modelo que recientemente fue publiado por Nvidia, Kumo Tabular, que justamente busca resolver problemas de tablas, ya sea de regresion o clasificación.

Entre sus ventajas es poder obtener predicciones sin la necesidad de tanto feature engineering. Este fue entrenado en miles de tablas sinteticas.

Ademas, suele dar un buen resultado predictivo sin necesidad de tuning y mayor data engineering.

Este modelo viene con una idea similar a los LLM.

In [ ]:
# Imports here and not in setup: this section needs the optional "kumo" group (PyTorch + CUDA)
import torch
import sdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
kumo = sdm.models.KumoTabular(task="regression", size="large", device=device)

def to_table(X, y=None):
    """pandas -> Kumo table. The target is NaN for the rows to predict."""
    df = X.assign(target=np.nan if y is None else y.to_numpy()).astype("float32")
    stypes = sdm.infer_stypes(df, overrides={"target": "numerical"})
    return sdm.TableTensor.from_pandas(df=df, stypes=stypes, device=device)

def kumo_predict(X_context, y_context, X_query, batch_size=4096):
    """Labeled rows go in as context; returns the mean of the 999 predicted quantiles."""
    context, query = to_table(X_context, y_context), to_table(X_query)
    predictions = []
    with torch.amp.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
        kumo.fit(x=context.drop_columns("target"), y=context[:, "target"])
        for start in range(0, len(X_query), batch_size):
            quantiles = kumo.predict(query[start:start + batch_size].drop_columns("target"))
            predictions.append(quantiles.to_pandas().to_numpy().mean(axis=1))
    kumo.clear()
    return np.concatenate(predictions)

In [ ]:
# Same 10 folds as the other models: each fold's train part is the context
cv_rmse = []
for train_idx, val_idx in kf.split(X_train):
    fold_pred = kumo_predict(X_train.iloc[train_idx], y_train.iloc[train_idx], X_train.iloc[val_idx])
    cv_rmse.append(np.sqrt(mean_squared_error(y_train.iloc[val_idx], fold_pred)))
cv_rmse = np.array(cv_rmse)

y_pred_kumo = kumo_predict(X_train, y_train, X_test_plain)
kumo_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_kumo))

model_comparison.loc["Kumo Tabular"] = {
    "RMSE CV": cv_rmse.mean(),
    "RMSE CV (std)": cv_rmse.std(),
    "RMSE train": np.nan,
    "RMSE test": kumo_test_rmse,
    "Ratio test/train": np.nan,
    "R² test": r2_score(y_test, y_pred_kumo),
}
model_comparison.sort_values("RMSE test")

El RMSE de CV (1.345) y el de test (1.359) son prácticamente iguales, por lo que Kumo Tabular generaliza bien y no hay señales de overfitting. Asi que el experimento tuvo buenos resultados, un modelo con mejores predicciones incluso que los modelos tipo arbol.

## 9. Conclusiones

### 9.1 Hallazgos

- El modelo final (OLS, 41 variables) explica cerca del 49% de la variación en reseñas: R² ajustado 0.502 en train, R² 0.488 en test.
- No hay overfitting: RMSE de 1.701 en train y 1.723 en test (ratio 1.013), y el CV de 10 folds da 1.702 ± 0.011.
- Backward y forward llegaron a las mismas 41 variables. Windows se quitó por criterio, porque casi todos los juegos lo tienen y la constante absorbe su efecto.
- Efectos más grandes, manteniendo lo demás constante:
  - has_metacritic +237%
  - mature +148%
  - cat_online_co_op +104%
  - is_free +82%
  - has_press_quotes +69%
  - duplicar los DLC +59%
  - cada año desde el lanzamiento +31%
- Efectos negativos: has_notes −32%, cat_family_sharing −29%, genre_casual −28%, self_published −21%.
- Modelos extra:
  - Ridge y Lasso dan lo mismo que OLS (Lasso no anuló ningún coeficiente).
  - Los árboles llegan a R² ≈ 0.62.
  - Kumo Tabular llega a 0.682.

### 9.2 Limitaciones

- Asociación, no causalidad. Por ejemplo, Metacritic no causa reseñas: los juegos grandes son los que reciben crítica.
- La mitad de la variación queda sin explicar. El dataset no tiene marketing, calidad real, reputación del estudio ni wishlists.
- Las reseñas son un proxy de popularidad, no de ventas. Además existen el review bombing y los juegos gratis.
- Multicolinealidad: signos negativos en Linux, cat_pvp, cat_co_op, cat_shared_split_screen y cat_steam_leaderboards, aunque su correlación individual era positiva.
- Error multiplicativo: e^RMSE = 5.6×, así que una predicción típica puede fallar por ese factor. Los juegos exitosos se subestiman (regresión a la media) y algunas predicciones quedan por debajo de 0.
- Causalidad inversa en DLC count: los juegos populares sacan más DLC.
- Alcance de los datos:
  - Se mantuvieron productos de software (Utilities, Design & Illustration).
  - Solo se usaron juegos con owners distintos de "0 - 0".
  - Es una foto del dataset v55 (referencia: 2026-03-01).
- Con n ≈ 70k casi todo p-value sale significativo, por eso se complementó con R² ajustado y la prueba F.
- Kumo y Gradient Boosting predicen mejor pero no son interpretables: no tienen ecuación ni coeficientes.

### 9.3 Recomendaciones

- Para desarrolladores (como asociaciones, no garantías): conseguir cobertura de crítica y prensa, traducir a más idiomas, incluir co-op online, soporte de control y Steam Cloud.
- Para trabajo futuro:
  - Agregar variables nuevas: historial del estudio calculado sin leakage, fecha y temporada de lanzamiento, historial de precios, texto de la descripción.
  - Modelar reseñas por año en lugar del total.
  - Usar GB o Kumo cuando importe más predecir que interpretar.

### 9.4 Respuesta al Objetivo

- ¿Qué hace popular a un juego? Se asocia a tres cosas:
  - visibilidad y escala de producción (crítica, prensa, clasificación mature)
  - tiempo y contenido (años en el mercado, DLC, logros)
  - alcance (idiomas, plataformas, multijugador online)
- Un modelo lineal interpretable explica ~49%. Modelos no lineales llegan hasta ~68%, a cambio de perder la interpretación.

## 10. Referencias

código (puede ser un link), tablas complementarias, etc.